<a href="https://colab.research.google.com/github/ojaspaul123/Fake-News-Detector-System/blob/main/fake_news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
from google.colab import files
uploaded = files.upload()
df_true = pd.read_csv('True.csv')
df_true.head()

Saving True.csv to True.csv


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [4]:
df_true.shape

(21417, 4)

In [5]:
from google.colab import files
uploaded = files.upload()
df_fake = pd.read_csv('Fake.csv')
df_fake.head()

Saving Fake.csv to Fake.csv


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [6]:
df_fake.shape

(23481, 4)

In [7]:
df_true['class'] = 1
df_fake['class'] = 0

In [8]:
df_true.shape , df_fake.shape

((21417, 5), (23481, 5))

In [9]:
df_true.head(10)

,title,text,subject,date,class
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1
5,"White House, Congress prepare for talks on spe...","WEST PALM BEACH, Fla./WASHINGTON (Reuters) - T...",politicsNews,"December 29, 2017",1
6,"Trump says Russia probe will be fair, but time...","WEST PALM BEACH, Fla (Reuters) - President Don...",politicsNews,"December 29, 2017",1
7,Factbox: Trump on Twitter (Dec 29) - Approval ...,The following statements were posted to the ve...,politicsNews,"December 29, 2017",1
8,Trump on Twitter (Dec 28) - Global Warming,The following statements were posted to the ve...,politicsNews,"December 29, 2017",1
9,Alabama official to certify Senator-elect Jone...,WASHINGTON (Reuters) - Alabama Secretary of St...,politicsNews,"December 28, 2017",1


In [10]:
df_merge = pd.concat([df_true,df_fake],axis=0)
df_merge.head()

,title,text,subject,date,class
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


In [11]:
df_merge.columns

Index(['title', 'text', 'subject', 'date', 'class'], dtype='object')

In [12]:
df_merge = df_merge.fillna(' ')

In [13]:
df_merge.isnull().sum()

,0
title,0
text,0
subject,0
date,0
class,0


In [14]:
df_merge['content'] = df_merge['subject']+' '+ df_merge['title']
df_merge.head()

,title,text,subject,date,class,content
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1,"politicsNews As U.S. budget fight looms, Repub..."
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1,politicsNews U.S. military to accept transgend...
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1,politicsNews Senior U.S. Republican senator: '...
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1,politicsNews FBI Russia probe helped by Austra...
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1,politicsNews Trump wants Postal Service to cha...


In [15]:
df_merge = df_merge.sample(frac=1)


In [16]:
df_merge.reset_index(inplace=True)
df_merge.drop(['index'],axis=1,inplace=True)

In [17]:
df_merge.head()

,title,text,subject,date,class,content
0,Scientists identify remains of 88 Argentine so...,BUENOS AIRES (Reuters) - Forensic scientists h...,worldnews,"December 1, 2017",1,worldnews Scientists identify remains of 88 Ar...
1,PRESIDENT TRUMP Receives Patriots Jersey From ...,In an outdoor ceremony to celebrate the super ...,left-news,"Apr 19, 2017",0,left-news PRESIDENT TRUMP Receives Patriots Je...
2,TRUMP MAKES BIG ANNOUNCEMENT After FOX News Sa...,Do you agree with Trump s decision? Will Trump...,politics,"Jan 26, 2016",0,politics TRUMP MAKES BIG ANNOUNCEMENT After FO...
3,"Trump again urges Senate to loosen rules, push...",WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"May 30, 2017",1,politicsNews Trump again urges Senate to loose...
4,Baby banned from Japanese municipal assembly,TOKYO (Reuters) - A baby brought into a Japane...,worldnews,"November 24, 2017",1,worldnews Baby banned from Japanese municipal ...


In [18]:
X= df_merge.drop('class',axis=1)
y= df_merge['class']

In [19]:
print(X)

                                                   title  \
0      Scientists identify remains of 88 Argentine so...   
1      PRESIDENT TRUMP Receives Patriots Jersey From ...   
2      TRUMP MAKES BIG ANNOUNCEMENT After FOX News Sa...   
3      Trump again urges Senate to loosen rules, push...   
4           Baby banned from Japanese municipal assembly   
...                                                  ...   
44893  Flames raced along train at west London statio...   
44894  Three arrested in Malaysia for suspected beer ...   
44895  BENGHAZI DAD Rips Into Hillary: Stood By My So...   
44896   Federal Court Strikes Down Racist Texas Voter...   
44897  Facebook's Zuckerberg to meet conservatives on...   

                                                    text       subject  \
0      BUENOS AIRES (Reuters) - Forensic scientists h...     worldnews   
1      In an outdoor ceremony to celebrate the super ...     left-news   
2      Do you agree with Trump s decision? Will Trump... 

In [20]:
ps = PorterStemmer()
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [ps.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content



In [21]:
df_merge['content'] = df_merge['content'].apply(stemming)

In [22]:
df_merge['content']

,content
0,worldnew scientist identifi remain argentin so...
1,left news presid trump receiv patriot jersey c...
2,polit trump make big announc fox news say keep...
3,politicsnew trump urg senat loosen rule push h...
4,worldnew babi ban japanes municip assembl
...,...
44893,worldnew flame race along train west london st...
44894,worldnew three arrest malaysia suspect beer fe...
44895,polit benghazi dad rip hillari stood son caske...
44896,news feder court strike racist texa voter id law


In [23]:
X= df_merge['content'].values
y = df_merge['class'].values

In [24]:
vector = TfidfVectorizer()
vector.fit(X)
X = vector.transform(X)



In [25]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 463407 stored elements and shape (44898, 13208)>
  Coords	Values
  (0, 549)	0.4026260143041836
  (0, 4115)	0.5359322501118406
  (0, 5697)	0.3902288199995346
  (0, 9595)	0.34329018061218075
  (0, 10215)	0.3995852117116278
  (0, 10825)	0.32594258142056987
  (0, 13018)	0.12097655984066871
  (1, 1891)	0.4114311919596395
  (1, 2200)	0.3215372973834156
  (1, 4581)	0.33870337496653546
  (1, 5572)	0.20702105503593646
  (1, 6144)	0.3825902159892931
  (1, 6632)	0.16434051923735693
  (1, 7874)	0.10165703428865049
  (1, 8517)	0.36787651645821734
  (1, 9001)	0.2104402780544939
  (1, 9441)	0.3605370951546983
  (1, 12011)	0.1078595866642543
  (1, 12599)	0.1349270451285284
  (1, 12888)	0.21586796442188957
  (2, 444)	0.2869065796469213
  (2, 1112)	0.28584012294696326
  (2, 2918)	0.2778107092755023
  (2, 4515)	0.2651120519042615
  (2, 6314)	0.2935011179549275
  :	:
  (44895, 6726)	0.2195814814925789
  (44895, 8850)	0.11693953405506567
  (4489

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify=y, random_state=2)

In [27]:
X_train.shape

(35918, 13208)

In [28]:
model = LogisticRegression()
model.fit(X_train,y_train)

LogisticRegression()

In [29]:
train_y_pred = model.predict(X_train)
print(accuracy_score(train_y_pred,y_train))

0.9999443176123393


In [30]:
testing_y_pred = model.predict(X_test)
print(accuracy_score(testing_y_pred,y_test))

0.9998886414253898


In [87]:
input_data = X_test[391]
prediction = model.predict(input_data)

In [88]:
if prediction[0] == 0:
    print('The News Is Fake')
else:
    print('The News is Real')



The News Is Fake


In [89]:
df_merge['content'][391]

'govern news epic commi obama pictur vietnam presid front'

In [34]:
df_merge['content'][42]

'polit valeri jarrett claim obama presid scandal free list obama top scandal prove lie video'

In [36]:
import joblib

joblib.dump(model, 'model.pkl')
joblib.dump(vector, 'vector.pkl')
print("Saved")

Saved
